# 🫡 蘇蘇指揮官 v5.0LB — 誠實回測引擎版 (Google Colab)

**數據源：Yahoo Finance（免費，無需帳號）**

### 與舊版分別
舊版用簡單勝率門檻，會被**純噪音**呃（~16% 假股票都「通過」）。
v5.0 加入 **permutation 顯著性檢定 + FDR 多重比較校正**：
- 拒絕噪音：純隨機股票 FDR 後通過率 ≈ **0%**
- 捕捉真 edge：真實時機 edge 通過率 > **78%**
- **空結果係誠實**：冇 setup 通過 = 市場暫時冇可信 edge，唔係 bug

### 使用方法
1. **Runtime → Run all**（全部執行）
2. 喺最後一格修改你嘅持倉清單再跑

> ⚠️ 僅供學習研究，不構成投資建議。歷史顯著 ≠ 未來保證。


In [ ]:
!pip install yfinance pandas numpy --quiet

In [ ]:
%%writefile backtest_engine.py
# -*- coding: utf-8 -*-
"""
backtest_engine.py — 誠實回測引擎 v3

回應 code review 三大深層問題：
  1. backtest / live 對齊：trend_strong 逐根 K 線檢查（唔係淨係最後一日）
  2. in-sample 幻覺 / 多重比較：permutation test 計 p-value，再用 BH-FDR 校正
     —— 將「成條 K 線」打亂次序（destroy 時間結構，保留每根 bar 內部 OHLCV），
        跑同一個策略 N 次，睇真實 expectancy 有冇明顯高過「純運氣」。
  3. out-of-sample：報告最後 1/3 段嘅穩健度作參考。

統計設計重點：permutation 保留股票自己嘅 return 分佈（即漂移），
只打散「次序」。所以 p-value 問緊嘅係：
   「你個入場時機規則，有冇 value 過『隻股票本身升咗』？」
單純升嘅股票 → 時機規則加唔到 value → p≈0.5（正路，唔應該收貨）。

止蝕統一用收市價（permutation 後盤中高低位冇意義，要 apples-to-apples）。
經 validate_engine.py 驗證：純噪音 FDR 後通過率 ≈ 0（且 p<0.05 命中率 = 名義 5%），
有真實時機 edge 時通過率 > 80%。
"""

import numpy as np

NEG_INF = -1e18


# ----------------------------------------------------------------------
# 快速 numpy rolling
# ----------------------------------------------------------------------
def _roll_mean(a, w):
    c = np.cumsum(np.insert(a, 0, 0.0))
    out = np.full(len(a), np.nan)
    out[w - 1:] = (c[w:] - c[:-w]) / w
    return out


def _roll_std(a, w):
    c1 = np.cumsum(np.insert(a, 0, 0.0))
    c2 = np.cumsum(np.insert(a * a, 0, 0.0))
    out = np.full(len(a), np.nan)
    s = c1[w:] - c1[:-w]
    ss = c2[w:] - c2[:-w]
    var = (ss - s * s / w) / w
    out[w - 1:] = np.sqrt(np.maximum(var, 0.0))
    return out


def _roll_max(a, w):
    out = np.full(len(a), np.nan)
    for i in range(w - 1, len(a)):
        out[i] = a[i - w + 1:i + 1].max()
    return out


def _roll_min(a, w):
    out = np.full(len(a), np.nan)
    for i in range(w - 1, len(a)):
        out[i] = a[i - w + 1:i + 1].min()
    return out


def _atr(high, low, close, w=14):
    prev = np.roll(close, 1)
    prev[0] = close[0]
    tr = np.maximum.reduce([high - low, np.abs(high - prev), np.abs(low - prev)])
    return _roll_mean(tr, w)


# ----------------------------------------------------------------------
# 指標
# ----------------------------------------------------------------------
def compute_indicators(o, h, l, c, v):
    return {
        "ma20":  _roll_mean(c, 20),
        "ma50":  _roll_mean(c, 50),
        "ma150": _roll_mean(c, 150),
        "ma200": _roll_mean(c, 200),
        "std10": _roll_std(c, 10),
        "std50": _roll_std(c, 50),
        "volma5":  _roll_mean(v, 5),
        "volma50": _roll_mean(v, 50),
        "high250": _roll_max(c, 250),
        "low250":  _roll_min(c, 250),
        "atr": _atr(h, l, c, 14),
        "high": h, "low": l, "close": c, "volume": v,
    }


def trend_gate(ind):
    c = ind["close"]
    return (c > ind["ma50"]) & (ind["ma50"] > ind["ma150"]) & (ind["ma150"] > ind["ma200"])


# ----------------------------------------------------------------------
# 訊號產生器（全部逐根 K 線 gate）
# ----------------------------------------------------------------------
def sig_sniper(ind, trend, lo, hi):
    c, ma20 = ind["close"], ind["ma20"]
    out = []
    for i in range(lo, hi):
        if not trend[i] or np.isnan(ma20[i]) or np.isnan(ma20[i - 1]):
            continue
        if c[i] > ma20[i] and c[i - 1] <= ma20[i - 1]:
            out.append(i)
    return out


def sig_vcp(ind, trend, lo, hi):
    c, s10, s50 = ind["close"], ind["std10"], ind["std50"]
    v5, v50 = ind["volma5"], ind["volma50"]
    lo250, hi250 = ind["low250"], ind["high250"]
    out = []
    for i in range(lo, hi):
        if not trend[i]:
            continue
        if np.isnan(s10[i]) or np.isnan(s50[i]) or np.isnan(v5[i]) or np.isnan(v50[i]):
            continue
        if np.isnan(lo250[i]) or np.isnan(hi250[i]):
            continue
        pos = (c[i] > lo250[i] * 1.3) and (c[i] > hi250[i] * 0.75)
        if pos and s10[i] < s50[i] * 0.6 and v5[i] < v50[i]:
            out.append(i)
    return out


def sig_panic(ind, trend, lo, hi):
    c, h, l, atr = ind["close"], ind["high"], ind["low"], ind["atr"]
    out = []
    for i in range(lo, hi):
        pa = atr[i - 1]
        if np.isnan(pa) or pa <= 0:
            continue
        prev_range = h[i - 1] - l[i - 1]
        if prev_range > pa * 2 and c[i] > h[i - 1]:
            out.append(i)
    return out


SIGNALS = {"Sniper MA20": sig_sniper, "VCP 爆發": sig_vcp, "ATR Panic": sig_panic}


# ----------------------------------------------------------------------
# 回測（收市價止蝕）
# ----------------------------------------------------------------------
def backtest(close, sigs, hold=20, stop_pct=0.10):
    wins = total = 0
    win_r, loss_r = [], []
    n = len(close)
    for i in sigs:
        if i + hold >= n:
            continue
        entry = close[i]
        if close[i + 1:i + hold + 1].min() < entry * (1 - stop_pct):
            loss_r.append(-stop_pct)
            total += 1
            continue
        r = (close[i + hold] - entry) / entry
        (win_r if r > 0 else loss_r).append(r)
        wins += int(r > 0)
        total += 1
    if total == 0:
        return dict(win=0.0, count=0, exp=0.0, payoff=0.0)
    wr = wins / total
    aw = float(np.mean(win_r)) if win_r else 0.0
    al = float(abs(np.mean(loss_r))) if loss_r else 0.0
    payoff = (aw / al) if al > 0 else 0.0
    exp = (wr * aw - (1 - wr) * al) * 100
    return dict(win=round(wr * 100, 1), count=total,
                exp=round(exp, 3), payoff=round(payoff, 2))


def _run_once(o, h, l, c, v, sig_fn, lo, hi, hold, stop_pct):
    ind = compute_indicators(o, h, l, c, v)
    trend = trend_gate(ind)
    sigs = sig_fn(ind, trend, lo, hi)
    return backtest(c, sigs, hold, stop_pct)


# ----------------------------------------------------------------------
# Permutation：打亂整條 K 線次序（保留每根 bar 內部 OHLCV）
# ----------------------------------------------------------------------
def _permute_bars(o, h, l, c, v, rng):
    """以 log-return 重組：打散每日 return 次序，重建價格路徑；量同步打散。
    對 close-only 數據（O=H=L=C）等同於打散 return。"""
    n = len(c)
    log_ret = np.diff(np.log(c))
    perm = rng.permutation(n - 1)
    pr = log_ret[perm]
    new_c = c[0] * np.exp(np.concatenate([[0.0], np.cumsum(pr)]))
    # 用原 bar 嘅相對振幅套返落新 close（保留 intrabar 形狀）
    rel_h = np.divide(h, c, out=np.ones_like(c), where=c != 0)
    rel_l = np.divide(l, c, out=np.ones_like(c), where=c != 0)
    idx = np.concatenate([[0], perm + 1])     # 對應每個新 bar 用邊根原 bar 嘅形狀/量
    new_h = new_c * rel_h[idx]
    new_l = new_c * rel_l[idx]
    new_v = v[idx]
    new_o = new_c.copy()
    return new_o, new_h, new_l, new_c, new_v


def evaluate(o, h, l, c, v, sig_fn, hold=20, stop_pct=0.10,
             min_count=8, train_frac=0.66, n_perm=300, seed=0):
    n = len(c)
    warmup = 200
    lo = max(warmup, 50)
    hi = n - hold
    if hi - lo < 60:
        return None
    rng = np.random.default_rng(seed)

    full = _run_once(o, h, l, c, v, sig_fn, lo, hi, hold, stop_pct)

    split = max(lo + 1, int(n * train_frac))
    oos = _run_once(o, h, l, c, v, sig_fn, split, hi, hold, stop_pct) \
        if hi - split >= 20 else dict(win=0.0, count=0, exp=0.0, payoff=0.0)

    # 樣本不足就唔值得做 permutation（慳時間）；p=1.0 代表「無法證明有 edge」
    if full["count"] < min_count:
        return dict(
            win=full["win"], count=full["count"], exp=full["exp"], payoff=full["payoff"],
            oos_win=oos["win"], oos_count=oos["count"],
            oos_exp=oos["exp"], oos_payoff=oos["payoff"],
            p_value=1.0, passed_raw=False,
        )

    real_exp = full["exp"] if full["count"] >= 1 else NEG_INF
    null_exps = []
    for _ in range(n_perm):
        po, ph, pl, pc, pv = _permute_bars(o, h, l, c, v, rng)
        st = _run_once(po, ph, pl, pc, pv, sig_fn, lo, hi, hold, stop_pct)
        if st["count"] >= 1:
            null_exps.append(st["exp"])
    null_exps = np.array(null_exps) if null_exps else np.array([0.0])
    p_value = (np.sum(null_exps >= real_exp) + 1) / (len(null_exps) + 1)

    return dict(
        win=full["win"], count=full["count"], exp=full["exp"], payoff=full["payoff"],
        oos_win=oos["win"], oos_count=oos["count"],
        oos_exp=oos["exp"], oos_payoff=oos["payoff"],
        p_value=round(float(p_value), 4),
        passed_raw=bool(full["count"] >= min_count and full["exp"] > 0),
    )


# 方便 validate（close-only）
def evaluate_close(close, sig_fn=sig_sniper, **kw):
    c = np.asarray(close, dtype=float)
    return evaluate(c.copy(), c.copy(), c.copy(), c, np.ones_like(c), sig_fn, **kw)


# ----------------------------------------------------------------------
# 多重比較校正（Benjamini-Hochberg FDR）
# ----------------------------------------------------------------------
def bh_fdr(pvals, alpha=0.10):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=bool)
    order = np.argsort(p)
    ranked = p[order]
    passed = ranked <= alpha * (np.arange(1, m + 1) / m)
    if not passed.any():
        return np.zeros(m, dtype=bool)
    cutoff = ranked[np.max(np.where(passed)[0])]
    return p <= cutoff


In [ ]:
%%writefile soso_trader.py
# -*- coding: utf-8 -*-
"""
蘇蘇全自動投資助理 (Soso Trader Commander)
版本：v5.0LB (誠實回測引擎版)
功能：大市共振 + 板塊輪動 + 動能雷達 + 持倉檢查 + Sniper/VCP/Panic 嚴選 + BTC
數據：優先長橋 CLI，後備 Yahoo Finance
回測：permutation 顯著性檢定 + FDR 多重比較校正（見 backtest_engine.py）
      經 validate_engine.py 驗證：純噪音 FDR 後通過率 ≈ 0%，真實 edge > 78%
"""

import yfinance as yf
import pandas as pd
import numpy as np
import datetime as dt
import warnings
import subprocess
import json
import sys
import backtest_engine as be

warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1) 設定中心
# ==========================================
MAX_POSITIONS = 8

MY_HOLDINGS = ["TYL","TSLA","PLTR","GOOG","VT","AMAT","META","FIG"]
MY_PICKS    = ["FUTU","MU","JNJ","GE","GOOG","COST","MRVL","PLTR"]

SECTORS = {
    "XLK":"科技","XLF":"金融","XLV":"醫療","XLE":"能源","XLY":"非必需消費",
    "XLP":"必需消費","XLI":"工業","XLC":"通訊","XLU":"公用","SMH":"半導體"
}

SP100 = [
    "AAPL","ABBV","ABT","ACN","ADBE","AIG","AMD","AMGN","AMT","AMZN","AXP","BA","BAC","BK","BKNG",
    "BLK","BMY","BRK-B","C","CAT","CHTR","CL","CMCSA","COF","COP","COST","CRM","CSCO","CVS","CVX",
    "DE","DHR","DIS","DOW","DUK","EMR","EXC","F","FDX","GD","GE","GILD","GM","GOOG","GOOGL","GS",
    "HD","HON","IBM","INTC","JNJ","JPM","KHC","KO","LIN","LLY","LMT","LOW","MA","MCD","MDLZ","MDT",
    "MET","META","MMM","MO","MRK","MS","MSFT","NEE","NFLX","NKE","NVDA","ORCL","PEP","PFE","PG",
    "PM","PYPL","QCOM","RTX","SBUX","SCHW","SO","SPG","T","TGT","TMO","TMUS","TSLA","TXN","UNH",
    "UNP","UPS","USB","V","VZ","WFC","WMT","XOM"
]

UNIVERSE = list(set(SP100 + MY_PICKS) - set(MY_HOLDINGS))

HOLD_DAYS   = 20
STOP_PCT    = 0.10
MIN_COUNT   = 8       # 回測最少訊號數（太少冇統計意義）
SCAN_NPERM  = 400     # permutation 次數（決定 p-value 精度下限 ≈ 1/(N+1)）
FDR_ALPHA   = 0.10    # Benjamini-Hochberg 多重比較 FDR 水平
FRESH_BARS  = 2       # 訊號需喺最近 N 根 K 線出現先算「而家可入場」

# ==========================================
# 2) 長橋 CLI 數據層
# ==========================================
_LB_AVAILABLE = None

def lb_available():
    global _LB_AVAILABLE
    if _LB_AVAILABLE is None:
        try:
            r = subprocess.run(["longbridge", "check"], capture_output=True, timeout=5)
            _LB_AVAILABLE = r.returncode == 0
        except Exception:
            _LB_AVAILABLE = False
    return _LB_AVAILABLE

def to_lb_symbol(ticker):
    """Convert Yahoo ticker to Longbridge format."""
    if ticker.endswith(".HK"):
        return ticker
    special = {"BTC-USD": "BTC-USDT.OTC", "BRK-B": "BRK-B.US",
                "^VIX": "VIX.US", "^GSPC": "SPX.US", "^IXIC": "IXIC.US"}
    if ticker in special:
        return special[ticker]
    return f"{ticker}.US"

def lb_get_kline(ticker, period="day", count=500):
    """Fetch OHLCV from Longbridge CLI."""
    if not lb_available():
        return None
    sym = to_lb_symbol(ticker)
    if sym in ("VIX.US", "^VIX"):
        return None  # VIX not available via LB kline
    try:
        r = subprocess.run(
            ["longbridge", "kline", sym, "--period", period,
             "--count", str(count), "--format", "json"],
            capture_output=True, text=True, timeout=30
        )
        if r.returncode != 0:
            return None
        raw = json.loads(r.stdout)
        rows = raw if isinstance(raw, list) else raw.get("candles", raw.get("data", []))
        if not rows:
            return None
        df = pd.DataFrame(rows)
        # Normalize column names
        col_map = {}
        for col in df.columns:
            cl = col.lower()
            if cl in ("timestamp","time","date","datetime"): col_map[col] = "Date"
            elif cl == "open":  col_map[col] = "Open"
            elif cl == "high":  col_map[col] = "High"
            elif cl == "low":   col_map[col] = "Low"
            elif cl in ("close","adj_close"): col_map[col] = "Close"
            elif cl == "volume": col_map[col] = "Volume"
        df = df.rename(columns=col_map)
        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], unit="s", errors="coerce").fillna(
                pd.to_datetime(df["Date"], errors="coerce"))
            df = df.set_index("Date").sort_index()
        for col in ["Open","High","Low","Close","Volume"]:
            if col not in df.columns:
                df[col] = np.nan
        if "Adj Close" not in df.columns:
            df["Adj Close"] = df["Close"]
        return df.dropna(subset=["Close"])
    except Exception:
        return None

# ==========================================
# 3) 工具函數
# ==========================================
def get_data(ticker, period="2y"):
    # 1) Try Longbridge CLI
    count_map = {"1y": 260, "2y": 520, "3y": 780, "3mo": 65}
    lb_count = count_map.get(period, 520)
    df = lb_get_kline(ticker, "day", lb_count)
    if df is not None and len(df) >= 25:
        return df

    # 2) Fallback: Yahoo Finance
    try:
        df = yf.download(ticker, period=period, progress=False, auto_adjust=False)
        if df is None or df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if "Adj Close" in df.columns:
            df["Close"] = df["Adj Close"]
        if "Close" not in df.columns:
            return None
        df = df.dropna()
        return df if len(df) >= 25 else None
    except Exception:
        return None

def calc_indicators(df):
    df = df.copy()
    df["MA20"]  = df["Close"].rolling(20).mean()
    df["MA50"]  = df["Close"].rolling(50).mean()
    df["MA150"] = df["Close"].rolling(150).mean()
    df["MA200"] = df["Close"].rolling(200).mean()
    df["High250"] = df["Close"].rolling(250).max()
    df["Low250"]  = df["Close"].rolling(250).min()

    delta = df["Close"].diff()
    up = delta.clip(lower=0)
    down = -1 * delta.clip(upper=0)
    rs = up.rolling(14).mean() / (down.rolling(14).mean() + 1e-12)
    df["RSI"] = 100 - (100 / (1 + rs))

    h, l, c = df["High"], df["Low"], df["Close"]
    prev_c = c.shift(1)
    tr = pd.concat([(h - l), (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    df["ATR"] = tr.rolling(14).mean()

    df["STD10"]   = df["Close"].rolling(10).std()
    df["STD50"]   = df["Close"].rolling(50).std()
    df["VolMA5"]  = df["Volume"].rolling(5).mean()
    df["VolMA50"] = df["Volume"].rolling(50).mean()
    return df

# 回測 / 顯著性檢定已搬入 backtest_engine.py（誠實回測引擎 v3）：
#   • 逐根 K 線 trend gate（修正 backtest/live 對齊）
#   • permutation test 計 p-value（真實時序 vs 打散時序）
#   • Benjamini-Hochberg FDR 校正多重比較
# 經 validate_engine.py 驗證：純噪音 FDR 後通過率 ≈ 0%，真實 edge > 78%。

# ==========================================
# 4) 模組
# ==========================================
def check_market_resonance():
    print("🔍 檢查大市共振 (QQQ+SPY+VIX)...")
    qqq = get_data("QQQ", "1y")
    spy = get_data("SPY", "1y")
    vix = get_data("^VIX", "1y")

    if qqq is None or spy is None or vix is None:
        return "數據不足", "⚠️ 無法判斷", False, np.nan

    qqq = calc_indicators(qqq)
    spy = calc_indicators(spy)

    q_p    = float(qqq["Close"].iloc[-1])
    q_ma50 = float(qqq["MA50"].iloc[-1])
    q_ma200= float(qqq["MA200"].iloc[-1])
    s_p    = float(spy["Close"].iloc[-1])
    s_ma50 = float(spy["MA50"].iloc[-1])
    v_p    = float(vix["Close"].iloc[-1])

    if q_p < q_ma200:
        return "🔴 熊市 (Risk Off)", f"QQQ跌穿年線，現金為王 (VIX:{v_p:.2f})", False, v_p

    score = sum([q_p > q_ma50, s_p > s_ma50])
    if score == 2: return "🟢 全面 Risk On", f"趨勢強勁 (VIX:{v_p:.2f})", True, v_p
    elif score == 1: return "🟡 震盪 (Neutral)", f"分歧市況 (VIX:{v_p:.2f})", True, v_p
    else: return "🔴 轉弱 (Warning)", f"動能減弱 (VIX:{v_p:.2f})", False, v_p

def check_sectors():
    print("📊 掃描板塊輪動中...")
    sector_perf = []
    for ticker, name in SECTORS.items():
        df = get_data(ticker, "1y")
        if df is None or len(df) < 25:
            continue
        ret = (float(df["Close"].iloc[-1]) / float(df["Close"].iloc[-20]) - 1) * 100
        sector_perf.append({"Code": ticker, "Name": name, "Perf": ret})

    if not sector_perf:
        return "N/A", "N/A"

    df_sec = pd.DataFrame(sector_perf).sort_values("Perf", ascending=False)
    top_str  = ", ".join([r["Name"] for _, r in df_sec.head(3).iterrows()])
    weak_str = ", ".join([r["Name"] for _, r in df_sec.tail(3).iterrows()])
    return top_str, weak_str

def check_momentum():
    print("⚡ 掃描全場動能排行 (5日/20日)...")
    mom_data = []
    for t in UNIVERSE:
        df = get_data(t, "3mo")
        if df is None or len(df) < 25:
            continue
        p = float(df["Close"].iloc[-1])
        ret_5d  = (p / float(df["Close"].iloc[-5])  - 1) * 100
        ret_20d = (p / float(df["Close"].iloc[-20]) - 1) * 100
        mom_data.append({"Ticker": t, "5d%": round(ret_5d,1), "20d%": round(ret_20d,1)})

    if not mom_data:
        return pd.DataFrame(), pd.DataFrame()

    df_mom = pd.DataFrame(mom_data)
    return df_mom.sort_values("20d%", ascending=False).head(5), \
           df_mom.sort_values("20d%", ascending=True).head(3)

def check_holdings():
    print("💼 檢查持倉健康度...")
    report = []
    current_count = len(MY_HOLDINGS)

    if current_count >= MAX_POSITIONS:
        discipline_msg = f"⚠️ 持倉爆額 ({current_count}/{MAX_POSITIONS})！❌ 暫時停止買入，只准賣出。"
        can_buy = False
    else:
        discipline_msg = f"✅ 額度正常 ({current_count}/{MAX_POSITIONS})，可尋找機會。"
        can_buy = True

    for t in MY_HOLDINGS:
        df = get_data(t, "1y")
        if df is None:
            continue
        df = calc_indicators(df)
        p    = float(df["Close"].iloc[-1])
        ma20 = float(df["MA20"].iloc[-1])
        ma50 = float(df["MA50"].iloc[-1])

        if np.isnan(ma20) or np.isnan(ma50):
            continue

        if p > ma20:   status, act = "🟢 強勢", "持有"
        elif p > ma50: status, act = "🟡 回調", "觀察"
        else:          status, act = "🔴 轉弱", "止蝕/減倉"

        report.append([t, status, round(p,2), act])

    return pd.DataFrame(report, columns=["Ticker","Status","Price","Action"]), discipline_msg, can_buy

def _ohlcv(df):
    """由 DataFrame 抽 numpy OHLCV；缺欄位用收市價/1 補上。"""
    c = df["Close"].astype(float).values
    h = df["High"].astype(float).values  if "High"   in df.columns else c.copy()
    l = df["Low"].astype(float).values   if "Low"    in df.columns else c.copy()
    o = df["Open"].astype(float).values  if "Open"   in df.columns else c.copy()
    v = df["Volume"].astype(float).values if "Volume" in df.columns else np.ones_like(c)
    return o, h, l, c, v


def run_scan():
    """
    嚴選掃描：誠實回測 + permutation 顯著性 + FDR 多重比較校正。
    只考慮「而家可入場」（訊號喺最近 FRESH_BARS 根 K 線出現）嘅 setup，
    對佢哋嘅歷史 edge 做 permutation test，最後喺整個候選家族做 FDR 校正。
    回傳：候選 list、掃描組合數、FDR 顯著數。
    """
    print("🚀 嚴選掃描中：誠實回測 + permutation 顯著性 + FDR 校正"
          f"（n_perm={SCAN_NPERM}, FDR={FDR_ALPHA}）...")
    candidates = []
    n_combos = 0

    for t in UNIVERSE:
        df = get_data(t, "3y")
        if df is None or len(df) < 280:
            continue
        o, h, l, c, v = _ohlcv(df)
        if np.isnan(c[-1]):
            continue
        ind = be.compute_indicators(o, h, l, c, v)
        trend = be.trend_gate(ind)
        n = len(c)
        price = float(c[-1])
        atr = float(ind["atr"][-1])
        if np.isnan(atr) or atr <= 0:
            continue

        for name, fn in be.SIGNALS.items():
            n_combos += 1
            # 而家可入場？訊號喺最近 FRESH_BARS 根 K 線出現
            sigs_now = fn(ind, trend, max(200, 50), n)
            if not any(s >= n - FRESH_BARS for s in sigs_now):
                continue
            res = be.evaluate(o, h, l, c, v, fn,
                              hold=HOLD_DAYS, stop_pct=STOP_PCT,
                              min_count=MIN_COUNT, n_perm=SCAN_NPERM, seed=42)
            if res is None or not res["passed_raw"]:
                continue
            stop = round(float(ind["low"][-1]), 2) if name == "ATR Panic" \
                else round(price - 2 * atr, 2)
            candidates.append({
                "Ticker": t, "Setup": name, "Price": round(price, 2),
                "Win": res["win"], "Count": res["count"], "Exp%": res["exp"],
                "Payoff": res["payoff"], "p": res["p_value"], "Stop": stop,
            })

    sig_n = 0
    if candidates:
        mask = be.bh_fdr(np.array([d["p"] for d in candidates]), FDR_ALPHA)
        for d, m in zip(candidates, mask):
            d["顯著"] = "✅" if m else "—"
        sig_n = int(mask.sum())
    return candidates, n_combos, sig_n

def check_btc():
    print("🪙 檢查 Crypto...")
    df = get_data("BTC-USD", "2y")
    if df is None:
        return "N/A", "N/A"
    df_w = df.resample("W").last()
    if len(df_w) < 60:
        return f"${float(df_w['Close'].iloc[-1]):,.0f}", "N/A"
    ma50 = float(df_w["Close"].rolling(50).mean().iloc[-1])
    p    = float(df_w["Close"].iloc[-1])
    halving = dt.datetime(2024, 4, 20)
    days = (dt.datetime.now() - halving).days
    status = "🐂 牛市" if p > ma50 else "🐻 震盪"
    return f"${p:,.0f}", f"{status} | 減半後 {days} 天"

# ==========================================
# 5) 主程式
# ==========================================
def main():
    print("\n" + "="*58)
    print(f"🫡 蘇蘇指揮官報告 v5.0LB | {dt.date.today()}")
    if lb_available():
        print("   📡 數據源：長橋 CLI ✅ (已登入)")
    else:
        print("   📡 數據源：Yahoo Finance (長橋未登入)")
    print("="*58)

    m_status, m_msg, market_ok, vix_val = check_market_resonance()
    print(f"\n🌡️ 大市：{m_status}\n   👉 {m_msg}")

    top, weak = check_sectors()
    print(f"\n📊 板塊：🔥 {top} | ❄️ {weak}")

    df_top_mom, df_weak_mom = check_momentum()
    print(f"\n⚡ 動能雷達：")
    if not df_top_mom.empty:
        print(f"   📈 最強 Top 5 (20日)：{', '.join(df_top_mom['Ticker'].tolist())}")
    if not df_weak_mom.empty:
        print(f"   📉 最弱 Top 3 (20日)：{', '.join(df_weak_mom['Ticker'].tolist())}")

    df_h, disc_msg, allow_buy = check_holdings()
    print(f"\n💼 持倉狀態：\n   {disc_msg}")
    if not df_h.empty:
        print(df_h.to_string(index=False, header=False))

    if market_ok:
        if not allow_buy:
            print("\n👀 提示：持倉已滿，以下只供「眼看手勿動」(Window Shopping)。")

        cands, n_combos, sig_n = run_scan()
        print(f"\n🔬 掃描統計：{n_combos} 個(股票×策略)組合 → "
              f"{len(cands)} 個過預過濾 → {sig_n} 個經 FDR 統計顯著。")

        if not cands:
            print("   (冇 setup 通過預過濾)")
        else:
            dfc = pd.DataFrame(cands).sort_values("p", ascending=True)
            sigdf = dfc[dfc["顯著"] == "✅"]
            rawdf = dfc[dfc["顯著"] == "—"]
            cols = ["Ticker","Setup","Price","Win","Count","Exp%","Payoff","p","Stop"]
            if not sigdf.empty:
                print("\n🎯【統計顯著・跑贏運氣】(permutation + FDR 通過，可信度最高)")
                print(sigdf[cols].to_string(index=False))
            else:
                print("\n🟡 今日冇任何 setup 通過統計顯著校正。")
                print("   意思：呢啲 setup 嘅歷史表現，分唔清係實力定運氣。")
                print("   建議：觀望 / 保留現金。空結果係誠實，唔係 bug。")
            if not rawdf.empty:
                print("\n📋 (只過預過濾、未達統計顯著 —— 僅參考，唔建議行動)")
                print(rawdf[["Ticker","Setup","Price","Exp%","Payoff","p"]].head(8).to_string(index=False))
    else:
        print("\n🛑 掃描暫停：大市風險高 (紅燈)，保留現金。")

    bp, bs = check_btc()
    print(f"\n🪙 BTC：{bp} | {bs}")

    print("\n" + "="*58)
    print("🧘‍♂️ 任務完成：深呼吸兩下，呼氣一下；放低電話，陪老婆囡囡。")
    print("="*58)

if __name__ == "__main__":
    main()


## ▶️ 執行（可喺下面改持倉清單）

In [ ]:
import soso_trader as st

# ============== ⚙️ 你嘅個人設定（改呢度）==============
st.MY_HOLDINGS = ["TYL", "TSLA", "PLTR", "GOOG", "VT", "AMAT", "META", "FIG"]
st.MY_PICKS    = ["FUTU", "MU", "JNJ", "GE", "GOOG", "COST", "MRVL", "PLTR"]
st.UNIVERSE    = list(set(st.SP100 + st.MY_PICKS) - set(st.MY_HOLDINGS))

# permutation 次數：越大 p-value 越精細但越慢（Colab 建議 300）
st.SCAN_NPERM  = 300
# =====================================================

st.main()
